# Grid Reconfiguration Sample Viewer

Inspect one downloaded `samples_XXbus.csv` file at a time. The left plot shows the existing connection state; the right plot shows the updated/reconfigured state. Use **Previous** and **Next** to move through rows. Hover nodes and lines to see their names and status.


In [1]:
from pathlib import Path
import ast

import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output


In [2]:
# Pick the CSV you want to inspect.
# Change this path to samples_37bus.csv, samples_69bus.csv, samples_84bus.csv, or samples_136bus.csv as needed.
CSV_PATH = Path("/Users/town/Codes/LLM4DistReconfig/Dataset/samples_33bus.csv")

# Plot size in pixels.
FIG_HEIGHT = 760
FIG_WIDTH = 1200


In [3]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {CSV_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(2)


Loaded /Users/town/Codes/LLM4DistReconfig/Dataset/samples_33bus.csv
Rows: 17,520
Columns: ['buses', 'lines', 'line_impedances', 'existing_connectivitty', 'existing_open_lines', 'existing_node_voltages', 'existing_system_loss', 'system_load', 'updated_connectivity', 'updated_open_lines', 'updated_node_voltages', 'updated_system_loss']


,buses,lines,line_impedances,existing_connectivitty,existing_open_lines,existing_node_voltages,existing_system_loss,system_load,updated_connectivity,updated_open_lines,updated_node_voltages,updated_system_loss
0,33,"[(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7...","[0.00064569, 0.00345195, 0.00256266, 0.0026684...","[[0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[(8, 21), (9, 15), (12, 22), (18, 33), (25, 29)]","[1.0, 0.999, 0.9945, 0.9922, 0.9901, 0.9847, 0...",19.4519,"[0j, (0.0333+0.02j), (0.0298+0.0132j), (0.0355...","[[0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[(14, 15), (32, 33), (7, 8), (25, 29), (9, 10)]","[1.0, 0.999, 0.9956, 0.9943, 0.993, 0.9898, 0....",14.3490
1,33,"[(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7...","[0.00064569, 0.00345195, 0.00256266, 0.0026684...","[[0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[(14, 15), (32, 33), (7, 8), (25, 29), (9, 10)]","[1.0, 0.9989, 0.9948, 0.9928, 0.9908, 0.9854, ...",27.8035,"[0j, (0.032+0.0192j), (0.0426+0.0189j), (0.029...","[[0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[(14, 15), (7, 8), (28, 29), (31, 32), (9, 10)]","[1.0, 0.9989, 0.995, 0.9944, 0.994, 0.9929, 0....",25.4783


In [4]:
def parse_value(value):
    """Parse list/tuple/complex values stored as strings in the CSV."""
    if isinstance(value, str):
        return ast.literal_eval(value)
    return value


def edge_key(edge):
    """Normalize an undirected edge so (i, j) and (j, i) compare equal."""
    i, j = edge
    return tuple(sorted((int(i), int(j))))


def closed_edges(all_lines, open_lines):
    open_set = {edge_key(edge) for edge in open_lines}
    return [edge for edge in all_lines if edge_key(edge) not in open_set]


def make_layout(lines, bus_count):
    """Create one stable layout from the full candidate topology."""
    graph = nx.Graph()
    graph.add_nodes_from(range(1, bus_count + 1))
    graph.add_edges_from(lines)
    return nx.spring_layout(graph, seed=7, iterations=180)


def voltage_values(row, column):
    values = parse_value(row[column])
    return {idx + 1: float(value) for idx, value in enumerate(values)}


def edge_trace(edge, pos, *, color, width, dash, text, xaxis, yaxis):
    i, j = edge
    x0, y0 = pos[int(i)]
    x1, y1 = pos[int(j)]
    return go.Scatter(
        x=[x0, x1],
        y=[y0, y1],
        mode="lines",
        line=dict(color=color, width=width, dash=dash),
        hoverinfo="text",
        text=[text, text],
        showlegend=False,
        xaxis=xaxis,
        yaxis=yaxis,
    )


def load_values(row):
    loads = parse_value(row["system_load"])
    return {idx + 1: complex(value) for idx, value in enumerate(loads)}


def node_trace(bus_count, pos, voltages, loads, *, xaxis, yaxis):
    nodes = list(range(1, bus_count + 1))
    xs = [pos[node][0] for node in nodes]
    ys = [pos[node][1] for node in nodes]
    vs = [voltages.get(node, None) for node in nodes]
    texts = []
    for node in nodes:
        load = loads.get(node, 0j)
        voltage = voltages.get(node, float("nan"))
        parts = [
            f"node:{node}",
            f"voltage:{voltage:.4f} p.u.",
            f"load P:{load.real:.6g}",
            f"load Q:{load.imag:.6g}",
            f"load |S|:{abs(load):.6g}",
        ]
        if node == 1:
            parts.append("source/slack")
        texts.append("<br>".join(parts))
    sizes = [15 if node == 1 else 10 for node in nodes]
    outlines = ["#0969da" if node == 1 else "#24292f" for node in nodes]
    return go.Scatter(
        x=xs,
        y=ys,
        mode="markers+text",
        marker=dict(
            size=sizes,
            color=vs,
            colorscale="Viridis",
            colorbar=dict(title="voltage<br>p.u.", x=1.02),
            line=dict(color=outlines, width=1.5),
            showscale=True,
        ),
        text=[str(node) for node in nodes],
        textposition="top center",
        textfont=dict(size=9),
        hoverinfo="text",
        hovertext=texts,
        showlegend=False,
        xaxis=xaxis,
        yaxis=yaxis,
    )


def add_state(fig, *, col, title, lines, open_lines, voltages, loads, system_loss, pos, bus_count):
    xaxis = "x" if col == 1 else "x2"
    yaxis = "y" if col == 1 else "y2"
    open_set = {edge_key(edge) for edge in open_lines}

    for idx, edge in enumerate(lines):
        i, j = map(int, edge)
        is_open = edge_key(edge) in open_set
        status = "open" if is_open else "closed"
        color = "#d1242f" if is_open else "#57606a"
        width = 3.0 if is_open else 1.7
        dash = "dash" if is_open else "solid"
        text = f"line:{i}->{j}<br>index:{idx}<br>status:{status}"
        fig.add_trace(edge_trace(edge, pos, color=color, width=width, dash=dash, text=text, xaxis=xaxis, yaxis=yaxis), row=1, col=col)

    fig.add_trace(node_trace(bus_count, pos, voltages, loads, xaxis=xaxis, yaxis=yaxis), row=1, col=col)
    fig.layout.annotations[col - 1].text = f"{title}<br><sup>open lines: {len(open_lines)} | system loss: {system_loss}</sup>"


def draw_sample(index):
    row = df.iloc[index]
    bus_count = int(row["buses"])
    lines = parse_value(row["lines"])
    existing_open = parse_value(row["existing_open_lines"])
    updated_open = parse_value(row["updated_open_lines"])
    existing_voltages = voltage_values(row, "existing_node_voltages")
    updated_voltages = voltage_values(row, "updated_node_voltages")
    loads = load_values(row)
    pos = make_layout(lines, bus_count)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Existing connection state", "Updated connection state"),
        horizontal_spacing=0.04,
    )
    add_state(
        fig,
        col=1,
        title="Existing connection state",
        lines=lines,
        open_lines=existing_open,
        voltages=existing_voltages,
        loads=loads,
        system_loss=row["existing_system_loss"],
        pos=pos,
        bus_count=bus_count,
    )
    add_state(
        fig,
        col=2,
        title="Updated connection state",
        lines=lines,
        open_lines=updated_open,
        voltages=updated_voltages,
        loads=loads,
        system_loss=row["updated_system_loss"],
        pos=pos,
        bus_count=bus_count,
    )

    fig.update_layout(
        title=f"{CSV_PATH.name} | sample {index + 1:,} / {len(df):,}",
        height=FIG_HEIGHT,
        width=FIG_WIDTH,
        margin=dict(l=20, r=30, t=90, b=20),
        hovermode="closest",
        plot_bgcolor="white",
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
    fig.update_yaxes(visible=False, scaleanchor="x2", scaleratio=1, row=1, col=2)
    fig.show()


In [5]:
state = {"index": 0}
output = widgets.Output()
previous_button = widgets.Button(description="Previous", icon="arrow-left", button_style="")
next_button = widgets.Button(description="Next", icon="arrow-right", button_style="")
index_label = widgets.HTML()


def refresh():
    index_label.value = f"<b>Sample:</b> {state['index'] + 1:,} / {len(df):,}"
    previous_button.disabled = state["index"] <= 0
    next_button.disabled = state["index"] >= len(df) - 1
    with output:
        clear_output(wait=True)
        draw_sample(state["index"])


def go_previous(_):
    if state["index"] > 0:
        state["index"] -= 1
        refresh()


def go_next(_):
    if state["index"] < len(df) - 1:
        state["index"] += 1
        refresh()


previous_button.on_click(go_previous)
next_button.on_click(go_next)
controls = widgets.HBox([previous_button, next_button, index_label])
display(widgets.VBox([controls, output]))
refresh()
